## Unilingual Model Exploration

This section explores unilingual models (Ensemble methods) that uses one model per language


---
Note that cross-validation process differs if we use a multi-lingual model or mono-lingual model:
- Multi-Lingual: Each fold should contain all the nodes with the same sentence_id and for all languages! (To avoid unbalance)
- Uni-Lingual: Each fold should contain all the the nodes with the same sentence_id. There are 2 ways to do this:

In [2]:
from src.unilingual_ensemble import UnilingualEnsembleClassifier
from src.cross_validation import *

from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, AdaBoostClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 



train = pd.read_csv("./data/train_dataset_processed.csv")
test = pd.read_csv("./data/test_dataset_processed.csv")
train

,sentence_id,language,node,length,degree,avg_neighbor_deg,degree_squared,degree_diff,clustering,local_degree_ratio,max_neighbor_degree,degree_centrality,harmonic_centrality,betweenness_centrality,pagerank,root
0,2,Japanese,14,23,2,2.000000,4,0.000000,0,0.999995,2,0.090909,5.865512,0.173160,0.046568,0
1,2,Japanese,8,23,2,2.000000,4,0.000000,0,0.999995,2,0.090909,6.382179,0.246753,0.044352,0
2,2,Japanese,4,23,1,2.000000,1,-1.000000,0,0.499998,2,0.045455,4.561122,0.000000,0.027162,0
3,2,Japanese,6,23,2,2.000000,4,0.000000,0,0.999995,3,0.090909,5.823846,0.090909,0.048565,0
4,2,Japanese,2,23,3,1.666667,9,1.333333,0,1.799989,2,0.136364,6.991703,0.255411,0.066901,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197474,995,Russian,2,19,1,3.000000,1,-2.000000,0,0.333332,3,0.055556,5.302381,0.000000,0.030321,0
197475,995,Russian,14,19,1,5.000000,1,-4.000000,0,0.200000,5,0.055556,6.034524,0.000000,0.029739,0
197476,995,Russian,5,19,2,3.000000,4,-1.000000,0,0.666664,5,0.111111,6.701190,0.111111,0.057065,0
197477,995,Russian,16,19,1,2.000000,1,-1.000000,0,0.499998,2,0.055556,5.005159,0.000000,0.032147,0


### Model 1: Random Forest Ensemble

In [ ]:
# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=RandomForestClassifier,
    base_model_kwargs={'n_estimators': 100, 'n_jobs': 1},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid={
    'n_estimators': [250, 500],            # Number of trees
    'max_depth': [None, 10, 20],           # Tree depth; None allows full growth
    'class_weight': [None, 'balanced']         # For imbalanced classes
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

### Send this shit to kaggle

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()

# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,
    X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


## Model 2: XGBoost Ensemble

In [ ]:
# Imports
from src.unilingual_ensemble import UnilingualEnsembleClassifier
from src.cross_validation import run_groupkfold_cv
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load data
train = pd.read_csv("./data/train_dataset_processed.csv")
test = pd.read_csv("./data/test_dataset_processed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset

# Run GroupKFold CV on train
cv_scores = run_groupkfold_cv(
    X=train,
    y=train[target_col],
    group_colname=group_col,
    clf_cls=UnilingualEnsembleClassifier,
    clf_kwargs={
        'base_model_cls': XGBClassifier,
        'base_model_kwargs': {'n_estimators': 300, 'n_jobs': 1},
        'language_colname': 'language',
        'n_jobs': 8  # adjust based on your CPU
    },
    n_splits=5,
    metric_fn=accuracy_score,
    verbose=True
)

# Plot CV scores
plt.plot(cv_scores, marker='o')
plt.title('CV Accuracy Scores per Fold')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=XGBClassifier,
    base_model_kwargs={'n_estimators': 300, 'n_jobs': 1},
    language_colname='language',
    n_jobs=8
)
model.fit(train.drop(columns=target_col), train[target_col])

# Predict on test
test_preds = model.predict(test)

# If test has true labels, evaluate; otherwise just show predictions count
if target_col in test.columns:
    test_acc = accuracy_score(test[target_col], test_preds)
    print(f"Test Accuracy: {test_acc:.4f}")
else:
    print(f"Predicted {len(test_preds)} test samples.")

# Show a few predictions
print("Sample predictions:", test_preds[:10])


In [ ]:
# === Prepare Kaggle Submission === #

# Attach predictions to test DataFrame
test = test.copy()
test["root"] = test_preds  # 0 or 1

# Safety check: make sure 'node', 'language', and 'sentence_id' exist
required_cols = {'node', 'language', 'sentence_id'}
if not required_cols.issubset(test.columns):
    missing = required_cols - set(test.columns)
    raise ValueError(f"Missing required columns in test set: {missing}")

# Function to find root node per sentence group
def find_root(group):
    root_rows = group[group["root"] == 1]
    if not root_rows.empty:
        return root_rows.iloc[0]["node"]
    else:
        return 1  # fallback value if no root was predicted

# Group by language and sentence_id, apply root selector
roots = test.groupby(["language", "sentence_id"]).apply(find_root).reset_index(name="root")

# Add sequential 'id' column
roots.insert(0, "id", range(1, len(roots) + 1))

# Save submission file
submission = roots[["id", "root"]]
submission.to_csv("data/unilingual_xgb.csv", index=False)

print("Submission file saved to: data/unilingual_xgb.csv")
display(submission.head())


## Model 3: LightGBM Ensemble

In [12]:
# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=LGBMClassifier,
    base_model_kwargs={'n_estimators': 500, 'n_jobs': 1},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid={
    'max_depth': [None, 5, 10, 20],           # Tree depth; None allows full growth
    'class_weight': [None, 'balanced'],         # For imbalanced classes
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

[LightGBM] [Info] Number of positive: 250, number of negative: 3705
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000317 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4372
[LightGBM] [Info] Number of data points in the train set: 3955, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.063211 -> initscore=-2.695978
[LightGBM] [Info] Start training from score -2.695978
[LightGBM] [Info] Number of positive: 250, number of negative: 4063
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000389 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4497
[LightGBM] [Info] Number of data points in the train set: 4313, number of used features: 25
[LightGBM] [Info] [binary:

UnilingualEnsembleClassifier(base_model_cls=<class 'lightgbm.sklearn.LGBMClassifier'>,
                             base_model_kwargs={'n_estimators': 500,
                                                'n_jobs': 1},
                             gridsearch_per_language=True, n_jobs=21,
                             param_grid={'class_weight': [None, 'balanced'],
                                         'max_depth': [None, 5, 10, 20]})

In [13]:
from src.submission import generate_kaggle_submission
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_lightgbm.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


2025-05-29 13:45:28.580 | INFO     | src.submission:generate_kaggle_submission:29 - Generating predictions...
2025-05-29 13:45:38.604 | INFO     | src.submission:generate_kaggle_submission:42 - Detected multilingual (per-language model ensemble) setup.
2025-05-29 13:45:38.649 | SUCCESS  | src.submission:generate_kaggle_submission:75 - Submission saved to: data/predictions_submission_unilingual_lightgbm.csv


In [15]:
def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_lightgbm.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

Number of correct predictions: 2540 / 10395
Evaluation accuracy: 0.2443


## Model 4: SVM

In [ ]:
# Load data
from sklearn.svm import SVC
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=SVC,
    language_colname='language',
    cv=2,
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

In [17]:
from src.submission import generate_kaggle_submission
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_multilingual_svm.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


2025-05-29 13:53:04.069 | INFO     | src.submission:generate_kaggle_submission:29 - Generating predictions...
2025-05-29 13:53:04.070 | WARNING  | src.submission:generate_kaggle_submission:36 - Model does not support predict_proba; using predict() directly.


ValueError: could not convert string to float: 'Japanese'

In [ ]:
def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_lightgbm.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

### Model 5: eZAutoML

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from ezautoml.model import eZAutoML
from ezautoml.space.search_space import SearchSpace
from ezautoml.evaluation.metric import MetricSet, Metric
from ezautoml.evaluation.task import TaskType
from ezautoml.optimization.optimizers.random_search import RandomSearchOptimizer

# === Load your data ===
train_df = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test_df = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# === Set target and group columns ===
target_col = "root"
group_col = "sentence_id"  # not needed for this run, unless using grouped CV

# === Extract features and target ===
X = train_df.drop(columns=[target_col])
y = train_df[target_col]

# === Train/test split (optional if using full train set) ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# === Define metrics ===
metrics = MetricSet(
    {"accuracy": Metric(name="accuracy", fn=accuracy_score, minimize=False)},
    primary_metric_name="accuracy"
)

# === Define search space ===
search_space = SearchSpace.from_builtin("classification_space")

# === Initialize eZAutoML ===
ezautoml = eZAutoML(
    search_space=search_space,
    task=TaskType.CLASSIFICATION,
    metrics=metrics,
    max_trials=10,     # You can increase for better models
    max_time=600,      # 10 minutes
    seed=42
)

# === Fit the AutoML model ===
ezautoml.fit(X_train, y_train)

# === Evaluate ===
test_accuracy = ezautoml.test(X_test, y_test)
print("Test accuracy:", test_accuracy)

# === Summary of best models ===
ezautoml.summary(k=10)


ModuleNotFoundError: No module named 'torch'

## Model 6: HistogramBoosting

In [ ]:
# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=HistGradientBoostingClassifier,
    base_model_kwargs={'n_jobs': 1, "early_stopping": True},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid = {
        'max_depth': [None, 6, 10],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_iter': [100, 300, 500],
        'l2_regularization': [0.01, 0.1, 1],
        'min_samples_leaf': [20, 50, 100],
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

In [ ]:
from src.submission import generate_kaggle_submission
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_multilingual_svm.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


In [ ]:
def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")

try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_lightgbm.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")